# Stage 1: Exploratory Data Analysis (EDA)

This notebook covers the exploratory analysis of the Telco Customer Churn dataset. 
Our objectives are to inspect the dataset structure, address missing values, perform univariate/bivariate analysis, and extract actionable business insights.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Set style
sns.set_theme(style="whitegrid")
plt.rcParams.update({
    'font.size': 11,
    'axes.labelsize': 12,
    'axes.titlesize': 14,
    'xtick.labelsize': 10,
    'ytick.labelsize': 10
})

PRIMARY_COLOR = '#2A9D8F'
SECONDARY_COLOR = '#E76F51'
PALETTE = [PRIMARY_COLOR, SECONDARY_COLOR]

## 1. Load and Inspect Dataset

In [ ]:
df = pd.read_csv('../data/raw/telco_churn.csv')
print(f"Dataset Shape: {df.shape}")
df.info()

## 2. Check for Missing Values & Duplicates

In [ ]:
print("Duplicate rows:", df.duplicated().sum())
print("\nMissing values count per column:")
print(df.isnull().sum())

# Notice that TotalCharges is loaded as an object because it contains space characters
spaces_count = (df['TotalCharges'] == ' ').sum() + (df['TotalCharges'] == '').sum()
print(f"\nEmpty string or space values in TotalCharges: {spaces_count}")

## 3. Clean TotalCharges (Preprocessing for Analysis)

In [ ]:
# Replace empty spaces with NaN and cast to float
df['TotalCharges'] = df['TotalCharges'].replace(r'^\s*$', np.nan, regex=True)
df['TotalCharges'] = pd.to_numeric(df['TotalCharges'], errors='coerce')

# Fill missing TotalCharges with median
tc_median = df['TotalCharges'].median()
df['TotalCharges'] = df['TotalCharges'].fillna(tc_median)
print("Missing values in TotalCharges after cleaning:", df['TotalCharges'].isnull().sum())

## 4. Churn Distribution Analysis

In [ ]:
churn_counts = df['Churn'].value_counts()
churn_rate = (churn_counts['Yes'] / len(df)) * 100
print(f"Churn Counts:\n{churn_counts}")
print(f"\nChurn Rate: {churn_rate:.2f}%")

# Plot
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
axes[0].pie(churn_counts, labels=churn_counts.index, autopct='%1.1f%%', startangle=90, colors=PALETTE, explode=(0, 0.1), shadow=True)
axes[0].set_title('Churn Distribution %', fontweight='bold')

sns.barplot(x=churn_counts.index, y=churn_counts.values, ax=axes[1], palette=PALETTE, hue=churn_counts.index, legend=False)
axes[1].set_title('Customer Count by Churn Status', fontweight='bold')
axes[1].set_xlabel('Churn')
axes[1].set_ylabel('Count')
plt.tight_layout()
plt.show()

## 5. Correlation Heatmap

In [ ]:
numerical_cols = ['tenure', 'MonthlyCharges', 'TotalCharges']
df_corr = df[numerical_cols].copy()
df_corr['Churn_Encoded'] = df['Churn'].apply(lambda x: 1 if x == 'Yes' else 0)

plt.figure(figsize=(8, 6))
sns.heatmap(df_corr.corr(), annot=True, cmap='coolwarm', fmt=".3f", linewidths=0.5, vmin=-1, vmax=1)
plt.title('Correlation Matrix (Numerical Features & Churn)', fontweight='bold')
plt.tight_layout()
plt.show()

## 6. Tenure & Monthly Charges vs. Churn

In [ ]:
# Boxplot for Tenure
plt.figure(figsize=(10, 5))
sns.boxplot(data=df, x='Churn', y='tenure', palette=PALETTE, hue='Churn', legend=False)
plt.title('Distribution of Customer Tenure by Churn Status', fontweight='bold')
plt.xlabel('Churn')
plt.ylabel('Tenure (Months)')
plt.show()

# Violinplot for Monthly Charges
plt.figure(figsize=(10, 5))
sns.violinplot(data=df, x='Churn', y='MonthlyCharges', palette=PALETTE, hue='Churn', legend=False, inner='quartile')
plt.title('Distribution of Monthly Charges by Churn Status', fontweight='bold')
plt.xlabel('Churn')
plt.ylabel('Monthly Charges ($)')
plt.show()

## 7. Categorical Relations: Contract, Internet Service, and Payment Methods

In [ ]:
# Churn Rate by Contract
plt.figure(figsize=(10, 5))
contract_churn = df.groupby('Contract')['Churn'].value_counts(normalize=True).rename('percentage').reset_index()
contract_churn['percentage'] *= 100
sns.barplot(data=contract_churn, x='Contract', y='percentage', hue='Churn', palette=PALETTE)
plt.title('Churn Percentage by Contract Type', fontweight='bold')
plt.ylabel('Percentage (%)')
plt.show()

# Churn Rate by Payment Method
plt.figure(figsize=(12, 5))
payment_churn = df.groupby('PaymentMethod')['Churn'].value_counts(normalize=True).rename('percentage').reset_index()
payment_churn['percentage'] *= 100
sns.barplot(data=payment_churn, x='PaymentMethod', y='percentage', hue='Churn', palette=PALETTE)
plt.title('Churn Percentage by Payment Method', fontweight='bold')
plt.xticks(rotation=15, ha='right')
plt.ylabel('Percentage (%)')
plt.show()

# Churn Rate by Internet Service
plt.figure(figsize=(10, 5))
internet_churn = df.groupby('InternetService')['Churn'].value_counts(normalize=True).rename('percentage').reset_index()
internet_churn['percentage'] *= 100
sns.barplot(data=internet_churn, x='InternetService', y='percentage', hue='Churn', palette=PALETTE)
plt.title('Churn Percentage by Internet Service Type', fontweight='bold')
plt.ylabel('Percentage (%)')
plt.show()

## 8. Summary of Key EDA Findings

1. **Contract Type**: Month-to-month contracts have an extremely high churn risk (~42.7%). Two-year contracts have the lowest (~2.8%).
2. **Tenure**: Shorter tenure is highly associated with churn. Average tenure of churned customers is ~18 months compared to ~37.6 months for loyal customers.
3. **Monthly Charges**: Customers who churn have higher average monthly charges ($74.44 vs $61.27).
4. **Internet Service**: Fiber optic internet subscribers churn at a remarkably high rate (~41.9%) compared to DSL subscribers (~19.0%).
5. **Payment Method**: Electronic checks are associated with the highest churn rate (~45.3%). Automatic payment options (Credit Card / Bank Transfer) show significantly lower rates (~15-16%).